# Practice: Concurrency, Parallelism & Async IO

Write your solutions in the **YOUR CODE** cells, then run **Validate** cells.

These problems focus on `threading`, `concurrent.futures` and `asyncio` (they all run cleanly inside a notebook). `multiprocessing` is covered conceptually in the final problem, since spawning real processes from a notebook is unreliable on Windows.

In [1]:
# Run this cell once per notebook — provides test validation helpers.
import asyncio
import threading


def check_equal(name, actual, expected):
    """Compare actual vs expected; print pass/fail."""
    if actual == expected:
        print(f"✓ {name}")
        return True
    print(f"✗ {name}")
    print(f"  Expected: {expected!r}")
    print(f"  Got:      {actual!r}")
    return False


def run_tests(cases):
    """Run a list of (name, callable) tuples. Callable should return the answer."""
    passed = 0
    for name, fn in cases:
        try:
            if check_equal(name, fn(), True):
                passed += 1
        except Exception as exc:
            print(f"✗ {name} — Error: {type(exc).__name__}: {exc}")
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("🎉 All tests passed!")
    return passed == total


def run_async(coro):
    """Run a coroutine safely, even inside a notebook's running event loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    result = {}

    def _runner():
        result["value"] = asyncio.run(coro)

    t = threading.Thread(target=_runner)
    t.start()
    t.join()
    return result["value"]


---

## Problem 1: parallel squares `Easy`

Using `ThreadPoolExecutor`, define `square_all(nums)` that returns a list of each number squared, in the **same order** as the input. Use the pool's `map`.

**Examples**

```
square_all([1, 2, 3, 4]) → [1, 4, 9, 16]
```

In [ ]:
# YOUR CODE
from concurrent.futures import ThreadPoolExecutor

def square_all(nums):
    pass


In [ ]:
check_equal('basic', square_all([1, 2, 3, 4]), [1, 4, 9, 16])
check_equal('order', square_all([5, 1, 3]), [25, 1, 9])
check_equal('empty', square_all([]), [])


---

## Problem 2: thread-safe counter `Medium`

Define `count_up(num_threads, per_thread)` that starts `num_threads` threads, each incrementing a **shared** counter `per_thread` times. Protect the increment with a `Lock` so no updates are lost, then return the final total.

**Examples**

```
count_up(4, 1000) → 4000
```

In [ ]:
# YOUR CODE
from threading import Thread, Lock

def count_up(num_threads, per_thread):
    pass


In [ ]:
check_equal('4x1000', count_up(4, 1000), 4000)
check_equal('2x5000', count_up(2, 5000), 10000)


---

## Problem 3: worker queue `Medium`

Using `queue.Queue` and a few worker threads, define `process_items(items)` where each worker computes the square of an item. Return the **sum** of all squares. Order doesn't matter, but every item must be processed exactly once.

**Examples**

```
process_items([1, 2, 3]) → 14   # 1 + 4 + 9
```

In [ ]:
# YOUR CODE
from threading import Thread
import queue

def process_items(items):
    pass


In [ ]:
check_equal('small', process_items([1, 2, 3]), 14)
check_equal('range', process_items(list(range(1, 6))), 55)


---

## Problem 4: gather results `Medium`

Using `ThreadPoolExecutor.submit` and `as_completed`, define `fetch_lengths(words)` that returns a dict mapping each word to its length (pretend each lookup is slow I/O done in a thread).

**Examples**

```
fetch_lengths(['a', 'bb', 'ccc']) → {'a': 1, 'bb': 2, 'ccc': 3}
```

In [ ]:
# YOUR CODE
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_lengths(words):
    pass


In [ ]:
check_equal('basic', fetch_lengths(['a', 'bb', 'ccc']), {'a': 1, 'bb': 2, 'ccc': 3})
check_equal('empty', fetch_lengths([]), {})


---

## Problem 5: async gather `Medium`

Complete the coroutine `fetch(n)` so it does `await asyncio.sleep(0)` then returns `n * 2`. Then define the coroutine `double_all(nums)` that uses `asyncio.gather` to run `fetch` on every number concurrently and returns the results **in order**. Use the provided `run_async` helper to run it.

**Examples**

```
run_async(double_all([1, 2, 3])) → [2, 4, 6]
```

In [ ]:
# YOUR CODE
import asyncio

async def fetch(n):
    pass

async def double_all(nums):
    pass


In [ ]:
check_equal('basic', run_async(double_all([1, 2, 3])), [2, 4, 6])
check_equal('order', run_async(double_all([4, 0, 5])), [8, 0, 10])


---

## Problem 6: async timeout `Medium`

Define the coroutine `guard(coro, limit)` that returns the awaited result of `coro` if it finishes within `limit` seconds, otherwise returns the string `'timeout'`. Use `asyncio.wait_for` and catch `asyncio.TimeoutError`.

**Examples**

```
fast coroutine → its value;   slow coroutine → 'timeout'
```

In [ ]:
# YOUR CODE
import asyncio

async def guard(coro, limit):
    pass


In [ ]:
async def quick():
    await asyncio.sleep(0.01)
    return 'done'

async def slow():
    await asyncio.sleep(1)
    return 'done'

check_equal('fast', run_async(guard(quick(), 0.5)), 'done')
check_equal('slow', run_async(guard(slow(), 0.1)), 'timeout')


---

## Problem 7: choose the right tool `Easy`

From the GIL and decision-guide lessons, define `best_tool(workload)` that returns the recommended approach as a string:

- `'cpu'` (heavy computation) → `'multiprocessing'`
- `'io_few'` (a handful of I/O waits) → `'threading'`
- `'io_many'` (thousands of I/O waits) → `'asyncio'`

**Examples**

```
best_tool('cpu') → 'multiprocessing'
```

In [ ]:
# YOUR CODE
def best_tool(workload):
    pass


In [ ]:
check_equal('cpu', best_tool('cpu'), 'multiprocessing')
check_equal('io_few', best_tool('io_few'), 'threading')
check_equal('io_many', best_tool('io_many'), 'asyncio')
